In [38]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [39]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [40]:
df.head()

,price,sector,bedRoom,bathroom,agePossession,property_type,built_up_area,servant room,furnishing_type,luxury_category
0,15.50,sector 43,5,6,Moderately Old,house,5490.00,1,0,Medium
1,0.38,sohna road,2,2,Relatively New,flat,602.00,0,0,Low
2,0.70,sector 92,3,2,Relatively New,flat,1325.00,0,0,Medium
3,1.60,sector 102,3,3,Relatively New,flat,1315.00,0,0,Low
4,3.98,sector 66,3,4,Moderately Old,flat,2200.11,1,2,Medium


# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished

# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession


In [41]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [42]:
df.head()

,price,sector,bedRoom,bathroom,agePossession,property_type,built_up_area,servant room,furnishing_type,luxury_category
0,15.50,sector 43,5,6,old,house,5490.00,1,0,Medium
1,0.38,sohna road,2,2,new,flat,602.00,0,0,Low
2,0.70,sector 92,3,2,new,flat,1325.00,0,0,Medium
3,1.60,sector 102,3,3,new,flat,1315.00,0,0,Low
4,3.98,sector 66,3,4,old,flat,2200.11,1,2,Medium


In [43]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\Rahul Kuniyal\AppData\Local\Temp\ipykernel_17020\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [44]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

C:\Users\Rahul Kuniyal\AppData\Local\Temp\ipykernel_17020\2576796079.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


In [45]:
df.head()

,price,sector,bedRoom,bathroom,agePossession,property_type,built_up_area,servant room,furnishing_type,luxury_category
0,15.50,sector 43,5,6,old,1,5490.00,1,0,1
1,0.38,sohna road,2,2,new,0,602.00,0,0,0
2,0.70,sector 92,3,2,new,0,1325.00,0,0,1
3,1.60,sector 102,3,3,new,0,1315.00,0,0,0
4,3.98,sector 66,3,4,old,0,2200.11,1,2,1


In [46]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [47]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [48]:
y_log = np.log1p(y)

In [49]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [50]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [51]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [52]:
scores.mean(),scores.std()

(np.float64(0.8529991965665777), np.float64(0.016203449949199582))

In [53]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [54]:
lr.fit(X_scaled,y_log)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [55]:
lr.fit(X_scaled, y_log)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [56]:
lr.coef_

array([ 5.38787345e-02,  6.58344085e-02,  1.22317717e-01,  2.08691028e-01,
        5.04008231e-02,  7.18452447e-03,  5.42499340e-03,  1.01346919e-02,
       -2.23958994e-02, -3.09758072e-03, -3.84603731e-03, -5.01870699e-03,
        2.76896011e-02, -2.92966404e-03,  2.51676365e-03, -1.96591001e-02,
        9.60006583e-04, -1.25156111e-02,  1.73821827e-02,  2.64353728e-02,
        5.23534903e-03, -1.51606002e-02,  1.29348561e-02,  1.65822206e-02,
        3.12639283e-02,  3.27497667e-02, -1.95360730e-02, -1.24672805e-02,
        2.88084058e-02,  3.25741334e-03,  6.75277779e-03,  3.82285021e-03,
        1.33869543e-02,  6.78657562e-03, -9.50105683e-03,  3.64375487e-02,
        2.03693978e-03,  1.78731011e-02,  5.76214293e-02,  7.40774760e-02,
        6.93169529e-03,  4.20186600e-02, -1.28981437e-02, -1.08981300e-02,
       -1.42812449e-02,  6.46221597e-03,  2.24628067e-02,  2.62028298e-02,
       -6.97784836e-03,  1.08349693e-02, -2.42820864e-03, -1.10726448e-02,
        1.46939376e-03,  

In [57]:
ridge = Ridge(alpha=0.0001)

In [58]:
ridge.fit(X_scaled, y_log)

,alpha,0.0001
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [62]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,123),columns=X_scaled.columns).stack().reset_index().drop(columns = ['level_0']).rename(columns = {'level_1' :'feature', 0 : 'coef' })

In [63]:
coef_df

,feature,coef
0,bedRoom,0.053879
1,bathroom,0.065834
2,property_type,0.122318
3,built_up_area,0.208691
4,servant room,0.050401
...,...,...
118,sector_sector 9a,-0.005037
119,sector_sohna road,-0.027458
120,sector_sohna road road,-0.009411
121,agePossession_old,-0.007012


In [64]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.866
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     179.5
Date:                Wed, 08 Jul 2026   Prob (F-statistic):               0.00
Time:                        19:36:50   Log-Likelihood:                 597.38
No. Observations:                3554   AIC:                            -946.8
Df Residuals:                    3430   BIC:                            -181.0
Df Model:                         123                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 